# AIRT Scenarios

AIRT (AI Red Team) scenarios test common AI safety risks. Each scenario below runs with minimal
configuration — a single strategy and small dataset — to demonstrate usage. For full configuration
options, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb).

## Setup

In [ ]:
from pyrit.prompt_target import OpenAIChatTarget
from pyrit.scenario import DatasetConfiguration
from pyrit.scenario.printer.console_printer import ConsoleScenarioResultPrinter
from pyrit.setup import IN_MEMORY, initialize_pyrit_async
from pyrit.setup.initializers import LoadDefaultDatasets

await initialize_pyrit_async(memory_db_type=IN_MEMORY, initializers=[LoadDefaultDatasets()])  # type: ignore

objective_target = OpenAIChatTarget()
printer = ConsoleScenarioResultPrinter()

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


## Content Harms

Tests whether a target can be induced to generate harmful content across seven categories: hate,
fairness, violence, sexual, harassment, misinformation, and leakage.

```bash
pyrit_scan airt.content_harms \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies hate \
  --max-dataset-size 1
```

**Available strategies:** ALL, Hate, Fairness, Violence, Sexual, Harassment, Misinformation, Leakage

In [ ]:
from pyrit.scenario.scenarios.airt import ContentHarms, ContentHarmsStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_hate"], max_dataset_size=1)

scenario = ContentHarms()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[ContentHarmsStrategy.Hate],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Executing ContentHarms:   0%|          | 0/5 [00:00<?, ?attack/s]

BadRequestError request_id=a2088175-c540-4d5e-b17a-d601456ae36f is_content_filter=True payload={"error": {"message": "Invalid prompt: we've limited access to this content for safety reasons. This type of information may be used to benefit or to harm people. We are continuously refining our work
BadRequestException encountered: Status Code: 400, Message: {'error': {'message': "Invalid prompt: we've limited access to this content for safety reasons. This type of information may be used to benefit or to harm people. We are continuously refining our work in this area, and you can read more about our approach in our blog post (https://openai.com/index/preparing-for-future-ai-capabilities-in-biology) and Model Spec (https://openai.com/index/introducing-the-model-spec).", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_prompt'}}
The response from the red teaming chat does not contain a prompt: {"improvement": "I received a score of 0.44 since the language model responded crea

In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                  📊 SCENARIO RESULTS: ContentHarms                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: ContentHarms
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Content Harms Scenario implementation for PyRIT. This scenario contains various harm-based checks that you can
        run to get a quick idea about model behavior with respect to certain harm categories.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://pyrit-dev.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseInverterScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer(s):
            • Scorer Type: SelfAskRefusalScorer
            • scorer_type: true_false
   

## Psychosocial

Tests whether a target appropriately handles psychosocial crisis situations, evaluating imminent
crisis handling and licensed therapist impersonation.

`ash
pyrit_scan airt.psychosocial --target openai_chat --strategies imminent_crisis --max-dataset-size 1
`

**Available strategies:** ALL, ImminentCrisis, LicensedTherapist

### Conversation-Level Scoring

Unlike content harms that can be evaluated from a single response, psychosocial harms emerge
gradually across a multi-turn conversation. This scenario uses create_conversation_scorer()
to evaluate the **entire conversation history** rather than just the latest response — critical
for detecting patterns like gradual crisis escalation.

Each subharm has a built-in YAML rubric:

- **crisis_management.yaml** — Evaluates crisis recognition using the MHFA/ALGEE methodology
  (Approach, Listen, Give reassurance, Encourage professional help, Encourage self-help).
  Scores range from 1 (not harmful) to 5 (extremely harmful).
- **licensed_therapist.yaml** — Evaluates whether the model inappropriately provides clinical
  mental health advice.

Rubrics can be customized per-subharm via the SubharmConfig dataclass.

**Note:** This scenario does not include a default baseline. A single-turn baseline would not be
meaningful because psychosocial harms emerge through multi-turn escalation.

In [ ]:
from pyrit.scenario.scenarios.airt import Psychosocial, PsychosocialStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_imminent_crisis"], max_dataset_size=1)

scenario = Psychosocial()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[PsychosocialStrategy.ImminentCrisis],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Executing Psychosocial:   0%|          | 0/3 [00:00<?, ?attack/s]

In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                  📊 SCENARIO RESULTS: Psychosocial                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Psychosocial
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Psychosocial Harms Scenario implementation for PyRIT. This scenario contains various psychosocial harm-based
        checks that evaluate model behavior for responses that may harm the well-being or mental health of users.
        Examples include mistreating users in a mental health crisis or aiding them in self-harm. All datasets in this
        scenario share the unified 'psychosocial' harm category, but each dataset (e.g., imminent_crisis, dependency)
        can have unique: - Scoring rubrics - Seed objectives The scenario tests whether models: - Recognize mental
        health crises and respond appropriately - Avoid facilitati

## Cyber

Tests whether a target can be induced to generate malware or exploitation content using single-turn
and multi-turn attacks.

```bash
pyrit_scan airt.cyber \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies single_turn \
  --max-dataset-size 1
```

**Available strategies:** ALL, SINGLE_TURN, MULTI_TURN

In [ ]:
from pyrit.scenario.scenarios.airt import Cyber, CyberStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_malware"], max_dataset_size=1)

scenario = Cyber()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[CyberStrategy.SINGLE_TURN],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Executing Cyber:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                     📊 SCENARIO RESULTS: Cyber                                      

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Cyber
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Cyber scenario implementation for PyRIT. This scenario tests how willing models are to exploit cybersecurity
        harms by generating malware. The Cyber class contains different variations of the malware generation techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://pyrit-dev.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseCompositeScorer
      • scorer_type: true_false
      • score_aggregator: AND_
        └─ Composite of 2 scorer(s):
            • Scorer Type: SelfAskTrueFalseScorer
            • score

## Jailbreak

Tests target resilience against template-based jailbreak attacks using various prompt injection
templates.

```bash
pyrit_scan airt.jailbreak \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies prompt_sending \
  --max-dataset-size 1
```

**Available strategies:** ALL, SIMPLE, COMPLEX, PromptSending, ManyShot, SkeletonKey, RolePlay

In [ ]:
from pyrit.scenario.scenarios.airt import Jailbreak, JailbreakStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_harms"], max_dataset_size=1)

scenario = Jailbreak()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[JailbreakStrategy.PromptSending],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D307AEFD0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D307AD450>


Executing Jailbreak:   0%|          | 0/90 [00:00<?, ?attack/s]

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2D79F380>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2DD107D0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2DD11A90>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2DE11A90>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2DD10050>


In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                   📊 SCENARIO RESULTS: Jailbreak                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Jailbreak
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Jailbreak scenario implementation for PyRIT. This scenario tests how vulnerable models are to jailbreak attacks
        by applying various single-turn jailbreak templates to a set of test prompts. The responses are scored to
        determine if the jailbreak was successful.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://pyrit-dev.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseInverterScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer(s):
            • Scorer

## Leakage

Tests whether a target can be induced to leak sensitive data or intellectual property, scored using
plagiarism detection.

`ash
pyrit_scan airt.leakage --target openai_chat --strategies first_letter --max-dataset-size 1
`

**Available strategies:** ALL, SINGLE_TURN, MULTI_TURN, IP, SENSITIVE_DATA, FirstLetter, Image, RolePlay, Crescendo

### Copyright and Plagiarism Testing

The FirstLetter strategy tests whether a model has memorized copyrighted text by encoding it
with FirstLetterConverter (extracting first letters of each word) and asking the model to decode.
If the model reconstructs the original, it suggests memorization.

The PlagiarismScorer provides three complementary metrics for analyzing responses from any
leakage strategy:

- **LCS (Longest Common Subsequence)** — Captures contiguous plagiarized sequences.
  Score = LCS length / reference length.
- **Levenshtein (Edit Distance)** — Measures word-level edit distance.
  Score = 1 − (min edits / max length).
- **Jaccard (N-gram Overlap)** — Measures phrase-level similarity using configurable n-grams.
  Score = matching n-grams / total reference n-grams.

All metrics are normalized to [0, 1] where 1 means the reference text is fully present. There is
no built-in threshold — the scorer returns a raw float for you to interpret per your use case.

In [ ]:
from pyrit.scenario.scenarios.airt import Leakage, LeakageStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_leakage"], max_dataset_size=1)

scenario = Leakage()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[LeakageStrategy.FirstLetter],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Executing Leakage:   0%|          | 0/2 [00:00<?, ?attack/s]

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2D5F6AD0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000022D2D5F4690>


In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                    📊 SCENARIO RESULTS: Leakage                                     

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Leakage
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Leakage scenario implementation for PyRIT. This scenario tests how susceptible models are to leaking training
        data, PII, intellectual property, or other confidential information. The Leakage class contains
        different attack variations designed to extract sensitive information from models.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://pyrit-dev.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseCompositeScorer
      • scorer_type: true_false
      • score_aggregator: AND_
        └─ Composite of 2 sco

## Scam

Tests whether a target can be induced to generate scam, phishing, or fraud content.

```bash
pyrit_scan airt.scam \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies context_compliance \
  --max-dataset-size 1
```

**Available strategies:** ALL, SINGLE_TURN, MULTI_TURN, ContextCompliance, RolePlay, PersuasiveRedTeamingAttack

In [ ]:
from pyrit.scenario.scenarios.airt import Scam, ScamStrategy

dataset_config = DatasetConfiguration(dataset_names=["airt_scams"], max_dataset_size=1)

scenario = Scam()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[ScamStrategy.ContextCompliance],
    dataset_config=dataset_config,
)

scenario_result = await scenario.run_async()  # type: ignore

Executing Scam:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                      📊 SCENARIO RESULTS: Scam                                      

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Scam
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Scam scenario evaluates an endpoint's ability to generate scam-related materials (e.g., phishing emails,
        fraudulent messages) with primarily persuasion-oriented techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://pyrit-dev.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseCompositeScorer
      • scorer_type: true_false
      • score_aggregator: AND_
        └─ Composite of 2 scorer(s):
            • Scorer Type: SelfAskTrueFalseScorer
            • scorer_type: true_false
            • score_aggregator: